# Cross-encoder only — the robust ~30-min dictionary lever

Skips the fragile multi-hour finetune. Trains the IO↔EO cross-encoder on the
**existing committed candidates** and applies it — this is the part that
actually expands the dictionary (better retention of non-cognate true pairs).

Needs a **T4 GPU** for ~25 min. Finishes well inside one Colab session, so a
disconnect/recycle can't eat hours of work. Runtime → Change runtime type → T4.

## 1. Verify GPU (must show a T4 — if it errors, you're still in cooldown)

In [ ]:
!nvidia-smi

## 2. Clone + deps

In [ ]:
!git clone --depth=1 https://github.com/komapc/embedding-aligner.git
%cd embedding-aligner
!pip install -q -r requirements.txt

## 3. Fetch the cross-encoder positives (bilingual_raw + langlinks)

The candidates and EO vocab are already in the clone — only these two files
live as a release asset.

In [ ]:
REL = 'https://github.com/komapc/embedding-aligner/releases/download/cross-encoder-inputs'
!wget -q {REL}/cross_encoder_inputs.tar.gz
!mkdir -p extractor_work && tar xzf cross_encoder_inputs.tar.gz -C extractor_work --strip-components=1
!ls -lh extractor_work/ results/bert_ido_epo_alignment/translation_candidates.json

## 4. Train the cross-encoder  — ~25 min

`--surface-neg-per-pos 0` skips the slow CPU edit-distance negative pass
(it can grind for 30+ min with the GPU idle); the model trains on positives +
BERT hard negatives, which are the important signal. **Note the final held-out
F1 / AUC / precision** — they tell you where to set the apply threshold.

In [ ]:
!python3 scripts/16_train_cross_encoder.py \
  --bilingual-raw extractor_work/bilingual_raw.json \
  --langlinks extractor_work/io_eo_langlinks.json \
  --candidates results/bert_ido_epo_alignment/translation_candidates.json \
  --eo-vocab data/esperanto_vocabulary.txt \
  --model-out models/cross-encoder-io-eo \
  --epochs 3 --batch-size 32 --surface-neg-per-pos 0

## 5. Apply the cross-encoder → re-ranked pairs  — minutes

Raise `--threshold` if step 4's precision was modest (fewer, cleaner pairs).

In [ ]:
!python3 scripts/17_apply_cross_encoder.py \
  --model models/cross-encoder-io-eo \
  --candidates results/bert_ido_epo_alignment/translation_candidates.json \
  --output results/bert_ido_epo_alignment/translation_candidates_ce.json \
  --threshold 0.5 --top-k 3

## 6. Download outputs for the laptop-side finish

In [ ]:
!tar czf cross_encoder_outputs.tar.gz \
  results/bert_ido_epo_alignment/translation_candidates_ce.json \
  models/cross-encoder-io-eo
from google.colab import files
files.download('cross_encoder_outputs.tar.gz')